In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Libraries

The required libraries for embedding generation, vector indexing, and file handling are imported here. Keeping all imports together makes the notebook easier to manage and debug.

In [2]:
# ============================================================
# Section 2 : Import Required Libraries
# ============================================================

# Standard libraries
import json
import pickle
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd

# Progress bar
from tqdm.auto import tqdm

# Sentence Embeddings
from sentence_transformers import SentenceTransformer

# FAISS
!pip install faiss-cpu
import faiss

print("All required libraries imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.9 MB/s eta 0:00:00
All required libraries imported successfully.


# Project Configuration

The project paths and common settings used throughout this notebook are defined here. Keeping them in one place makes it easier to update the notebook without modifying multiple cells.

In [3]:
# ============================================================
# Section 3 : Project Configuration
# ============================================================

# Project root directory
PROJECT_ROOT = Path("/content/drive/MyDrive/AI_Loan_Advisory_Chatbot")

# Input directory
CHUNK_DIR = PROJECT_ROOT / "processed" / "chunks"

# Output directories
EMBEDDING_DIR = PROJECT_ROOT / "processed" / "embeddings"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db"

# Create output directories if they do not exist
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

# Embedding model
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Processing parameters
BATCH_SIZE = 32
RANDOM_SEED = 42

# Set random seed
np.random.seed(RANDOM_SEED)

print("=" * 60)
print("Project Configuration")
print("=" * 60)
print(f"Project Root       : {PROJECT_ROOT}")
print(f"Chunk Directory    : {CHUNK_DIR}")
print(f"Embeddings Folder  : {EMBEDDING_DIR}")
print(f"Vector DB Folder   : {VECTOR_DB_DIR}")
print(f"Embedding Model    : {EMBEDDING_MODEL_NAME}")
print(f"Batch Size         : {BATCH_SIZE}")
print("=" * 60)

Project Configuration
Project Root       : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot
Chunk Directory    : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/chunks
Embeddings Folder  : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/embeddings
Vector DB Folder   : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/vector_db
Embedding Model    : sentence-transformers/all-MiniLM-L6-v2
Batch Size         : 32


# Verify Project Structure

Before generating embeddings, the required input folders are checked. This helps ensure that the chunk files created in the previous notebook are available for processing.

In [4]:
# ============================================================
# Section 4 : Verify Project Structure
# ============================================================

def verify_project_structure(chunk_directory: Path):
    """Verify that the required project folders and chunk files exist."""

    if not chunk_directory.exists():
        raise FileNotFoundError(
            f"Chunk directory not found:\n{chunk_directory}\n\n"
            "Please complete Notebook 2 before running this notebook."
        )

    chunk_files = sorted(chunk_directory.rglob("chunks.json"))

    print("=" * 60)
    print("Project Structure Verification")
    print("=" * 60)
    print(f"Chunk Directory : {chunk_directory}")
    print(f"Chunk Files     : {len(chunk_files)}")
    print("=" * 60)

    if len(chunk_files) == 0:
        raise FileNotFoundError("No chunk files were found.")

    print("\nAvailable Chunk Files:\n")

    for file in chunk_files:
        relative_path = file.relative_to(chunk_directory)
        print(f"• {relative_path}")

    return chunk_files


# Verify folder structure
chunk_files = verify_project_structure(CHUNK_DIR)

Project Structure Verification
Chunk Directory : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/chunks
Chunk Files     : 9

Available Chunk Files:

• HDFC/Car Loan/chunks.json
• HDFC/Home Loan/chunks.json
• HDFC/Personal Loan/chunks.json
• SBI/Education Loan/chunks.json
• SBI/Gold Loan/chunks.json
• SBI/Home Loan/chunks.json
• SBI/Loan Against Property/chunks.json
• SBI/Personal Loan/chunks.json
• SBI/Vehicle Loan/chunks.json


# Load Chunk Files

The chunk files generated in the previous notebook are loaded and combined into a single collection. This creates a unified dataset that will be used for embedding generation in the following sections.

In [5]:
# ============================================================
# Section 5 : Load Chunk Files
# ============================================================

def load_chunk_files(chunk_files):
    """Load all chunk files into a single list."""

    all_chunks = []

    print("Loading chunk files...\n")

    for file_path in tqdm(chunk_files):

        with open(file_path, "r", encoding="utf-8") as file:
            chunks = json.load(file)

        all_chunks.extend(chunks)

    return all_chunks


# Load all chunks
all_chunks = load_chunk_files(chunk_files)

print("\n" + "=" * 60)
print("Chunk Loading Summary")
print("=" * 60)
print(f"Chunk Files Loaded : {len(chunk_files)}")
print(f"Total Chunks       : {len(all_chunks)}")
print("=" * 60)

Loading chunk files...



  0%|          | 0/9 [00:00<?, ?it/s]


Chunk Loading Summary
Chunk Files Loaded : 9
Total Chunks       : 241


# Validate Loaded Chunks

A few basic checks are performed to ensure that the loaded chunks have the expected structure. Any invalid entries are skipped before the embedding generation step.

In [6]:
# ============================================================
# Section 6 : Validate Loaded Chunks
# ============================================================

REQUIRED_FIELDS = [
    "chunk_id",
    "bank",
    "loan_type",
    "document_type",
    "section",
    "source_file",
    "page_number",
    "chunk_text",
    "word_count"
]


def validate_chunks(chunks):
    """Validate loaded chunk data."""

    valid_chunks = []

    missing_fields = 0
    empty_text = 0

    for chunk in chunks:

        # Check required fields
        if not all(field in chunk for field in REQUIRED_FIELDS):
            missing_fields += 1
            continue

        # Check chunk text
        text = chunk["chunk_text"]

        if not isinstance(text, str) or not text.strip():
            empty_text += 1
            continue

        valid_chunks.append(chunk)

    print("=" * 60)
    print("Chunk Validation Summary")
    print("=" * 60)
    print(f"Total Chunks        : {len(chunks)}")
    print(f"Valid Chunks        : {len(valid_chunks)}")
    print(f"Missing Fields      : {missing_fields}")
    print(f"Empty Chunk Text    : {empty_text}")
    print("=" * 60)

    return valid_chunks


# Validate chunks
valid_chunks = validate_chunks(all_chunks)

Chunk Validation Summary
Total Chunks        : 241
Valid Chunks        : 241
Missing Fields      : 0
Empty Chunk Text    : 0


# Load Embedding Model

The sentence embedding model is loaded once and reused for all chunks. A lightweight model is used to provide a good balance between retrieval quality and processing speed.

In [7]:
# ============================================================
# Section 7 : Load Embedding Model
# ============================================================

print("Loading embedding model...\n")

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

embedding_dimension = embedding_model.get_sentence_embedding_dimension()

print("=" * 60)
print("Embedding Model")
print("=" * 60)
print(f"Model Name          : {EMBEDDING_MODEL_NAME}")
print(f"Embedding Dimension : {embedding_dimension}")
print("=" * 60)
print("\nModel loaded successfully.")

Loading embedding model...



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model
Model Name          : sentence-transformers/all-MiniLM-L6-v2
Embedding Dimension : 384

Model loaded successfully.


/tmp/ipykernel_737/959089174.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = embedding_model.get_sentence_embedding_dimension()


# Generate Embeddings

Each chunk is converted into a dense vector representation using the selected sentence transformer model. Batch processing is used to improve performance while keeping memory usage manageable.

In [8]:
# ============================================================
# Section 8 : Generate Embeddings
# ============================================================

import time


def generate_embeddings(chunks, model, batch_size=32):
    """Generate embeddings for all chunk texts."""

    chunk_texts = [chunk["chunk_text"] for chunk in chunks]

    start_time = time.time()

    embeddings = model.encode(
        chunk_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False
    )

    processing_time = time.time() - start_time

    embeddings = embeddings.astype(np.float32)

    print("\n" + "=" * 60)
    print("Embedding Generation Summary")
    print("=" * 60)
    print(f"Total Chunks        : {len(chunks)}")
    print(f"Embedding Shape     : {embeddings.shape}")
    print(f"Embedding Dimension : {embeddings.shape[1]}")
    print(f"Processing Time     : {processing_time:.2f} seconds")
    print("=" * 60)

    return embeddings


# Generate embeddings
embeddings = generate_embeddings(
    valid_chunks,
    embedding_model,
    batch_size=BATCH_SIZE
)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embedding Generation Summary
Total Chunks        : 241
Embedding Shape     : (241, 384)
Embedding Dimension : 384
Processing Time     : 13.35 seconds


# Normalize Embeddings

The generated embeddings are normalized before indexing. This allows cosine similarity to be computed efficiently using the FAISS inner product index.

In [9]:
# ============================================================
# Section 9 : Normalize Embeddings
# ============================================================

def normalize_embeddings(embeddings):
    """Normalize embeddings for cosine similarity search."""

    normalized_embeddings = embeddings.copy()

    faiss.normalize_L2(normalized_embeddings)

    print("=" * 60)
    print("Embedding Normalization")
    print("=" * 60)
    print(f"Embedding Shape : {normalized_embeddings.shape}")
    print("Normalization   : Completed")
    print("=" * 60)

    return normalized_embeddings


# Normalize embeddings
normalized_embeddings = normalize_embeddings(embeddings)

Embedding Normalization
Embedding Shape : (241, 384)
Normalization   : Completed


# Build FAISS Index

A FAISS index is created using the normalized embeddings. The index stores the vector representations and enables fast similarity search during retrieval.

In [10]:
# ============================================================
# Section 10 : Build FAISS Index
# ============================================================

def build_faiss_index(embeddings):
    """Create a FAISS index using cosine similarity."""

    embedding_dimension = embeddings.shape[1]

    # Inner Product index (used with normalized embeddings)
    index = faiss.IndexFlatIP(embedding_dimension)

    # Add embeddings to the index
    index.add(embeddings)

    print("=" * 60)
    print("FAISS Index Summary")
    print("=" * 60)
    print(f"Index Type          : {type(index).__name__}")
    print(f"Embedding Dimension : {embedding_dimension}")
    print(f"Indexed Vectors     : {index.ntotal}")
    print("=" * 60)

    return index


# Build FAISS index
faiss_index = build_faiss_index(normalized_embeddings)

FAISS Index Summary
Index Type          : IndexFlatIP
Embedding Dimension : 384
Indexed Vectors     : 241


# Save Embeddings

The generated embeddings and their corresponding metadata are saved for later use. These files can be loaded directly during retrieval without repeating the embedding process.

In [11]:
# ============================================================
# Section 11 : Save Embeddings and Metadata
# ============================================================

def save_embeddings_and_metadata(embeddings, metadata, output_dir):
    """Save embeddings and metadata."""

    embeddings_path = output_dir / "embeddings.pkl"
    metadata_path = output_dir / "metadata.pkl"

    with open(embeddings_path, "wb") as file:
        pickle.dump(embeddings, file)

    with open(metadata_path, "wb") as file:
        pickle.dump(metadata, file)

    print("=" * 60)
    print("Embeddings Saved")
    print("=" * 60)
    print(f"Embeddings : {embeddings_path}")
    print(f"Metadata   : {metadata_path}")
    print("=" * 60)


save_embeddings_and_metadata(
    normalized_embeddings,
    valid_chunks,
    EMBEDDING_DIR
)

Embeddings Saved
Embeddings : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/embeddings/embeddings.pkl
Metadata   : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/embeddings/metadata.pkl


# Save FAISS Index

The FAISS index is stored on disk so it can be loaded directly during retrieval without rebuilding the vector database.

In [12]:
# ============================================================
# Section 12 : Save FAISS Index
# ============================================================

def save_faiss_index(index, metadata, vector_db_dir):
    """Save FAISS index and metadata."""

    index_path = vector_db_dir / "faiss_index.bin"
    metadata_path = vector_db_dir / "faiss_metadata.pkl"

    faiss.write_index(index, str(index_path))

    with open(metadata_path, "wb") as file:
        pickle.dump(metadata, file)

    print("=" * 60)
    print("FAISS Index Saved")
    print("=" * 60)
    print(f"Index    : {index_path}")
    print(f"Metadata : {metadata_path}")
    print("=" * 60)


save_faiss_index(
    faiss_index,
    valid_chunks,
    VECTOR_DB_DIR
)

FAISS Index Saved
Index    : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/vector_db/faiss_index.bin
Metadata : /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/vector_db/faiss_metadata.pkl


# Verify Saved Files

A final check is performed to confirm that all generated files have been saved successfully and are ready to be used by the retrieval pipeline.

In [13]:
# ============================================================
# Section 13 : Verify Saved Files
# ============================================================

def verify_saved_files():
    """Verify that all output files exist."""

    files = {
        "Embeddings": EMBEDDING_DIR / "embeddings.pkl",
        "Metadata": EMBEDDING_DIR / "metadata.pkl",
        "FAISS Index": VECTOR_DB_DIR / "faiss_index.bin",
        "FAISS Metadata": VECTOR_DB_DIR / "faiss_metadata.pkl"
    }

    print("=" * 60)
    print("Verification Summary")
    print("=" * 60)

    all_verified = True

    for name, path in files.items():
        exists = path.exists()
        status = "✓" if exists else "✗"
        print(f"{status} {name:<18} {path}")

        if not exists:
            all_verified = False

    print("=" * 60)

    if all_verified:
        print("All files saved successfully.")
    else:
        print("Some output files are missing.")


verify_saved_files()

Verification Summary
✓ Embeddings         /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/embeddings/embeddings.pkl
✓ Metadata           /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/processed/embeddings/metadata.pkl
✓ FAISS Index        /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/vector_db/faiss_index.bin
✓ FAISS Metadata     /content/drive/MyDrive/AI_Loan_Advisory_Chatbot/vector_db/faiss_metadata.pkl
All files saved successfully.
